In [2]:
def compute_pruning_ratios_from_amounts(amount_s, amount_u):
    """
    Convertit les 'amounts' de pruning en ratios globaux p_s et p_u.

    Hypothèse: pruning appliqué dans cet ordre:
      1) structured pruning avec amount_s
      2) unstructured pruning avec amount_u sur les poids restants

    Donc:
      p_s = amount_s
      p_u = (1 - p_s) * amount_u
    """
    if not (0.0 <= amount_s <= 1.0 and 0.0 <= amount_u <= 1.0):
        raise ValueError("amount_s et amount_u doivent être dans [0, 1].")

    p_s = float(amount_s)
    p_u = (1.0 - p_s) * float(amount_u)
    return p_s, p_u


def compute_effdl_score(amount_s, amount_u, q_w, q_a, w, f,
                        ref_params=5.6e6,
                        ref_ops=2.8e8):
    """
    Score EffDL en utilisant les amounts de pruning.

    On convertit d'abord:
      p_s = amount_s
      p_u = (1 - p_s) * amount_u

    Puis:
      score = [1 - (p_s + p_u)] * ((q_w / 32) * w) / ref_params
            + (1 - p_s) * (max(q_w, q_a) / 32) * f / ref_ops
    """
    p_s, p_u = compute_pruning_ratios_from_amounts(amount_s, amount_u)

    if min(q_w, q_a, w, f) < 0:
        raise ValueError("q_w, q_a, w et f doivent être non-négatifs.")

    params_term = (1.0 - (p_s + p_u)) * ((q_w / 32.0) * w) / ref_params
    ops_term = (1.0 - p_s) * (max(q_w, q_a) / 32.0) * f / ref_ops
    return params_term + ops_term


# Exemple
amount_s = 0.3
amount_u = 0.1
p_s, p_u = compute_pruning_ratios_from_amounts(amount_s, amount_u)
score = compute_effdl_score(amount_s=amount_s, amount_u=amount_u, q_w=8, q_a=8, w=7.2e5, f=4.5e7)

print(f"p_s (effectif) = {p_s:.4f}")
print(f"p_u (effectif) = {p_u:.4f}")
print(f"Score = {score:.4f}")

p_s (effectif) = 0.3000
p_u (effectif) = 0.0700
Score = 0.0484
